In [ ]:
import pandas as pd
import numpy as np
import json

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train      = X[X["moon"].isin(splits["train"])]
y_train      = y[y["moon"].isin(splits["train"])]
X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]


In [ ]:
import joblib
import os
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from scipy.stats import pearsonr


def train(X_train, y_train, model_directory_path, loop_moon, embargo):
    feature_cols = [c for c in X_train.columns if c.startswith("Feature_")]

    df = X_train.merge(y_train[["moon", "id", "target"]], on=["moon", "id"])
    df = df.dropna(subset=["target"])
    all_moons = np.array(sorted(df["moon"].unique()))
    n_moons   = len(all_moons)

    K           = 10
    N_QUINTILES = 5
    quintile    = (np.arange(n_moons) / n_moons * N_QUINTILES).astype(int)
    quintile    = np.clip(quintile, 0, N_QUINTILES - 1)

    models   = []
    fold_ics = []
    print(f"  Stratified {K}-fold, ALL {n_moons} moons, regularised LGBM params")

    for fold in range(K):
        train_idx, val_idx = [], []
        for q in range(N_QUINTILES):
            q_idx  = np.where(quintile == q)[0]
            n_val  = max(1, len(q_idx) // K)
            start  = (fold * n_val) % len(q_idx)
            v_pos  = np.arange(start, start + n_val) % len(q_idx)
            v_mask = np.zeros(len(q_idx), dtype=bool)
            v_mask[v_pos] = True
            val_idx.extend(q_idx[v_mask].tolist())
            train_idx.extend(q_idx[~v_mask].tolist())

        fold_tr_moons  = all_moons[train_idx].tolist()
        fold_val_moons = all_moons[val_idx].tolist()

        df_tr  = df[df["moon"].isin(fold_tr_moons)].copy()
        df_val = df[df["moon"].isin(fold_val_moons)].copy()

        # ── ONLY CHANGE vs Step 7: regularised params from Optuna trial 1 ─────
        m = LGBMRegressor(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,           # was 63 default → halved to 31 (best)
            min_child_samples=100,   # was 20 default → 5x more
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=fold,
            n_jobs=-1,
            verbose=-1,
        )
        # ── END OF CHANGE ─────────────────────────────────────────────────────

        m.fit(df_tr[feature_cols].fillna(0), df_tr["target"])

        corrs = []
        for moon in fold_val_moons:
            grp = df_val[df_val["moon"] == moon]
            if len(grp) < 2: continue
            r, _ = pearsonr(m.predict(grp[feature_cols].fillna(0)), grp["target"])
            if not np.isnan(r): corrs.append(r)
        ic = float(np.mean(corrs)) if corrs else 0.0
        fold_ics.append(ic)
        models.append(m)
        print(f"    Fold {fold+1}: train={len(fold_tr_moons)} moons  "
              f"val={len(fold_val_moons)} moons  IC={ic:+.5f}")

    print(f"  Mean fold IC = {np.mean(fold_ics):+.5f}")

    joblib.dump({"models": models, "features": feature_cols},
                os.path.join(model_directory_path, "model.joblib"))
    print("  Saved.")


In [ ]:
def infer(X_test, model_directory_path, loop_moon, embargo):
    obj      = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    models   = obj["models"]
    features = obj["features"]
    X        = X_test[features].fillna(0)
    preds    = np.mean([m.predict(X) for m in models], axis=0)

    return pd.DataFrame({
        "moon":       X_test["moon"].values,
        "id":         X_test["id"].values,
        "prediction": preds,
    })


In [ ]:
import os
from scipy.stats import pearsonr

embargo   = 4
model_dir = "./model"
os.makedirs(model_dir, exist_ok=True)

train(X_train, y_train, model_dir, loop_moon=0, embargo=embargo)

results = []
for moon in splits["reduced_cloud"]:
    pred = infer(X_test_cloud[X_test_cloud["moon"] == moon],
                 model_dir, loop_moon=moon, embargo=embargo)
    results.append(pred)

predictions = pd.concat(results)
corrs = []
for moon in splits["reduced_cloud"]:
    merged = predictions[predictions["moon"] == moon].merge(
        y_test_cloud[y_test_cloud["moon"] == moon], on=["moon", "id"])
    if len(merged) > 1:
        r, _ = pearsonr(merged["prediction"], merged["target"])
        corrs.append(r)
        print(f"Moon {moon}: Pearson r = {r:.4f}")
print(f"\nMean IC = {np.mean(corrs):.4f}")
